In [1]:
import pandas as pd
import numpy as np
import json
import re
import os
import yaml
import glob
from pathlib import Path
from datetime import datetime
import synapseclient
from synapseclient.models import (
    Column, ColumnType, Dataset, EntityRef, File, Folder, Project, FacetType,
    DatasetCollection, JSONSchema, SchemaOrganization,
)
import time
from pprint import pprint
from synapseclient.core.utils import make_bogus_data_file
from typing import Dict, List, Any, Set, Union
from synapseclient import Wiki
from collections import defaultdict, Counter
import warnings
warnings.filterwarnings('ignore')

syn = synapseclient.Synapse()
syn.login()



UPGRADE AVAILABLE

A more recent version of the Synapse Client (4.12.0) is available. Your version (4.11.0) can be upgraded by typing:
   pip install --upgrade synapseclient

Python Synapse Client version 4.12.0 release notes

https://python-docs.synapse.org/news/


Welcome, ram.ayyala!



In [2]:
def load_schema(schema_name: str, schema_dir: str = None) -> dict:
    """Load a JSON schema from the json-schemas directory by file name (without .json extension).

    Args:
        schema_name: Name of the schema file without extension (e.g. "ClinicalFile").
        schema_dir: Optional path to the json-schemas directory. If not provided,
                    searches for json-schemas/ relative to cwd or one level up.

    Returns:
        Parsed JSON schema as a dict.
    """
    if schema_dir is None:
        candidates = [Path("json-schemas"), Path("../json-schemas")]
        schema_dir = next((p for p in candidates if p.is_dir()), None)
        if schema_dir is None:
            raise FileNotFoundError(
                "Could not find json-schemas directory. "
                "Run from the repo root or pass schema_dir explicitly."
            )
    schema_path = Path(schema_dir) / f"{schema_name}.json"
    with open(schema_path) as f:
        return json.load(f)


## Helper Functions

The functions below encapsulate common Synapse patterns used throughout this tutorial. Run this cell once before executing any workflow cells.

In [ ]:
def setup_schema_organization(org_name: str) -> SchemaOrganization:
    """Get or create a Synapse JSON schema organization."""
    org = SchemaOrganization(name=org_name)
    try:
        org.store()
        print(f"Organization '{org_name}' created.")
    except Exception:
        org.get()
        print(f"Organization '{org_name}' already exists.")
    return org


def _bump_patch_version(version: str) -> str:
    """Increment the patch component of a semantic version (e.g. '0.0.1' → '0.0.2')."""
    parts = version.split(".")
    parts[-1] = str(int(parts[-1]) + 1)
    return ".".join(parts)


def _latest_semantic_version(schema: JSONSchema) -> Optional[str]:
    """Return the highest semantic version registered for a schema, or None if none found."""
    versions = []
    for v in schema.get_versions():
        sv = getattr(v, "semantic_version", None)
        if sv:
            try:
                versions.append(tuple(int(x) for x in sv.split(".")))
            except ValueError:
                pass
    if not versions:
        return None
    return ".".join(str(x) for x in max(versions))


def register_schema(
    schema_name: str,
    org_name: str,
    schema_body: dict,
    initial_version: str = "0.0.1",
) -> JSONSchema:
    """Register or update a JSON schema.

    - New schema: registered at `initial_version`.
    - Existing schema: patch version is bumped from the latest registered version.
    """
    schema = JSONSchema(name=schema_name, organization_name=org_name)
    try:
        schema.get()  # raises if schema doesn't exist yet
        latest = _latest_semantic_version(schema)
        new_version = _bump_patch_version(latest) if latest else initial_version
        schema.store(schema_body=schema_body, version=new_version)
        print(f"  '{schema_name}'  {latest or 'unversioned'} → {new_version}")
    except Exception:
        schema.store(schema_body=schema_body, version=initial_version)
        print(f"  '{schema_name}'  (new) → {initial_version}")
    return schema


def register_schemas(
    org_name: str,
    initial_version: str = "0.0.1",
    *,
    schema_name: str = None,
    schema_names: List[str] = None,
    schema_dir: str = None,
) -> Dict[str, JSONSchema]:
    """Register one, several, or all schemas from the json-schemas directory.

    Modes (pass exactly one):
      schema_name  — register a single schema by name
      schema_names — register a list of schemas by name
      (neither)    — register all non-empty .json files in schema_dir

    Returns:
        Dict mapping schema_name → JSONSchema.
    """
    if schema_dir is None:
        candidates = [Path("json-schemas"), Path("../json-schemas")]
        schema_dir = next((p for p in candidates if p.is_dir()), None)
        if schema_dir is None:
            raise FileNotFoundError("Could not find json-schemas directory.")
    schema_dir = Path(schema_dir)

    if schema_name:
        names = [schema_name]
    elif schema_names:
        names = list(schema_names)
    else:
        names = [
            p.stem for p in sorted(schema_dir.glob("*.json"))
            if p.stat().st_size > 0
        ]

    print(f"Registering {len(names)} schema(s) to '{org_name}':")
    results = {}
    for name in names:
        body = load_schema(name, schema_dir=str(schema_dir))
        results[name] = register_schema(name, org_name, body, initial_version)
    return results


def bind_schema_to_folder(folder: Folder, schema_uri: str) -> Folder:
    """Bind a registered JSON schema URI to a Synapse folder with derived annotations enabled."""
    bound = folder.bind_schema(json_schema_uri=schema_uri, enable_derived_annotations=True)
    info = bound.json_schema_version_info
    print(f"Schema bound: {info.id}")
    return folder


def get_validation_report(container) -> None:
    """Print a validation summary for a Folder or RecordSet.

    Folder   — get_schema_validation_statistics() + get_invalid_validation()
                 InvalidJSONSchemaValidation fields:
                   result.validation_response.object_id
                   result.validation_error_message
    RecordSet — validation_summary + get_detailed_validation_results() (DataFrame)
                  columns: row_index, is_valid, validation_error_message
    """
    if hasattr(container, "validation_summary"):
        # RecordSet path
        summary = container.validation_summary
        if summary:
            total   = summary.total_number_of_children
            valid   = summary.number_of_valid_children
            invalid = summary.number_of_invalid_children
        else:
            total = valid = invalid = "?"
        print(f"Validation summary — Total: {total} | Valid: {valid} | Invalid: {invalid}")
        df = container.get_detailed_validation_results()
        if df is not None:
            invalid_rows = df[df["is_valid"] == False]
            for _, row in invalid_rows.iterrows():
                print(f"  [row {row['row_index']}] {row['validation_error_message']}")
    else:
        # Folder path (JSON schema mixin)
        stats   = container.get_schema_validation_statistics()
        invalid = list(container.get_invalid_validation())
        total       = getattr(stats, "total_count", "?")
        valid_count = getattr(stats, "valid_count", "?")
        print(f"Validation summary — Total: {total} | Valid: {valid_count} | Invalid: {len(invalid)}")
        for result in invalid:
            obj_id = result.validation_response.object_id if result.validation_response else "?"
            print(f"  [{obj_id}] {result.validation_error_message}")


def run_grid_session(record_set, synapse_client) -> "Grid":
    """Open a new Grid session on a RecordSet and print instructions for the curator."""
    grid = Grid(record_set_id=record_set.id).create(synapse_client=synapse_client)
    print(f"Grid session ID: {grid.session_id}")
    print("→ Open the Synapse UI, navigate to the CurationTask, and fill in the Grid.")
    print("→ Then run the next cell to export and close the session.")
    return grid


In [4]:
# ── Project & Schema ─────────────────────────────────────────────────────────
PROJECT_NAME  = "Mock ALS Project"
ORG_NAME      = "ampals.schemas"
SCHEMA_NAME   = "ClinicalFile"   # filename in json-schemas/ without .json
VERSION       = "0.0.3"

# ── Curation Workflow ─────────────────────────────────────────────────────────
STAGING_FOLDER_ID     = "syn73452535"   # Synapse ID of the folder containing files to curate
CURATION_NAME         = "ClinicalFile_Staging"
CURATION_INSTRUCTIONS = (
    "Annotate each clinical file using the ALS Clinical File Schema. "
    "Fill in required fields: title, creator, keywords, source, url."
)
ATTACH_WIKI = True

# ── Derived (do not edit) ─────────────────────────────────────────────────────
PROJECT_ENT = Project(name=PROJECT_NAME).get()
SCHEMA_URI  = f"{ORG_NAME}-{SCHEMA_NAME}-{VERSION}"
schema_body = load_schema(SCHEMA_NAME)

print(f"Project:  {PROJECT_ENT.name} ({PROJECT_ENT.id})")
print(f"Schema:   {SCHEMA_URI}")
print(f"Staging:  {STAGING_FOLDER_ID}")


Project:  Mock ALS Project (syn69801668)
Schema:   ampals.schemas-ClinicalFile-0.0.3
Staging:  syn73452535


## Section 3: Schema Registration

Register the JSON schema with the Synapse organization. If the schema already exists at the given version, it will be retrieved rather than re-created.

After registration, you can optionally bind the schema to a test folder to verify the schema and check annotation validation.

In [5]:
org = setup_schema_organization(ORG_NAME)
schema = register_schema(SCHEMA_NAME, ORG_NAME, schema_body, VERSION)
print(f"Schema URI: {schema.uri}")


Organization 'ampals.schemas' already exists.
Schema 'ClinicalFile' already registered.
Schema URI: ampals.schemas-ClinicalFile


In [6]:
# Optional: bind the schema to a test folder to verify annotation structure
test_folder = Folder(name="test_folder", parent_id=PROJECT_ENT.id).store()
test_folder = bind_schema_to_folder(test_folder, schema.uri)
get_validation_report(test_folder)


Schema bound: ampals.schemas-ClinicalFile-0.0.1
Validation summary — Total: ? | Valid: ? | Invalid: 1


AttributeError: 'InvalidJSONSchemaValidation' object has no attribute 'object_id'

## Section 4: Curator Extension Workflows

The Synapse Curator extension provides a Grid-based spreadsheet interface where human curators fill in metadata fields defined by the JSON schema. Two workflow options are available:

- **Option A — File-Based**: Creates an EntityView of all files in a staging folder; curators annotate files directly via a Grid.
- **Option B — Record-Based**: Metadata is stored as a RecordSet table; structured records are created by curators and annotations can be propagated to individual files.

> **Note**: The Curator extension provides a schema-driven UI for human curators to enter and validate data. For AI-assisted annotation, see `enhance_annotations_with_ai()` in `synapse_dataset_manager.py`.

In [2]:
# Curator Extension Imports
from synapseclient.extensions.curator import (
    create_record_based_metadata_task,
    create_file_based_metadata_task,
)
from synapseclient.models.curation import CurationTask, Grid
from synapseclient.models import RecordSet

### Option A: File-Based Metadata Task

Creates an EntityView of all files in `test_folder` and opens a Grid session scoped to `schema.uri`. Curators fill in fields directly against existing Synapse files.

In [ ]:
entity_view_id, task_id = create_file_based_metadata_task(
    synapse_client=syn,
    folder_id=STAGING_FOLDER_ID,
    curation_task_name=f"{CURATION_NAME}_FileTask",
    instructions=CURATION_INSTRUCTIONS,
    attach_wiki=ATTACH_WIKI,
    entity_view_name=f"{CURATION_NAME}_View",
    schema_uri=SCHEMA_URI,
)
print(f"Entity View ID:   {entity_view_id}")
print(f"Curation Task ID: {task_id}")


### Option B: Record-Based Metadata Task

Creates a RecordSet table for structured metadata records. Returns the `RecordSet`, `CurationTask`, and an initial `Grid` session.

In [31]:
record_set, curation_task, data_grid = create_record_based_metadata_task(
    synapse_client=syn,
    project_id=PROJECT_ENT.id,
    folder_id=STAGING_FOLDER_ID,
    record_set_name=f"{CURATION_NAME}_Records",
    record_set_description=f"Metadata records for {STAGING_FOLDER_ID}",
    curation_task_name=f"{CURATION_NAME}_RecordTask",
    upsert_keys=["title"],
    instructions=CURATION_INSTRUCTIONS,
    schema_uri=SCHEMA_URI,
    bind_schema_to_record_set=True,
)
print(f"RecordSet ID:      {record_set.id}")
print(f"CurationTask ID:   {curation_task.task_id}")
print(f"Grid session ID:   {data_grid.session_id}")


Extracted schema properties and created template: ['url', 'title', 'sameAs', 'source', 'creator', 'disease', 'license', 'species', 'subject', 'citation', 'dataType', 'keywords', 'publisher', 'visitType', 'collection', 'studyPhase', 'annotations', 'contributor', 'description', 'keyMeasures', 'alternateName', 'analysisTypes', 'assessmentType', 'clinicalDomain', 'diseaseSubtype', 'subjectIdColumn', 'administrationMode', 'hasLongitudinalData', 'includedInDataCatalog']


Uploading to Synapse storage: 100%|██████████| 331/331 [00:00<00:00, 616B/s, tmppi4uglpq.csv]


Created RecordSet with ID: syn74385129 in folder syn73452535
Bound schema ampals.schemas-ClinicalFile-0.0.1 to RecordSet ID: syn74385129


[ERROR] Error creating CurationTask in Synapse: 404 Client Error: Entity syn74385059 is in trash can.


SynapseHTTPError: 404 Client Error: Entity syn74385059 is in trash can.

### List Existing Curation Tasks

Enumerate all CurationTasks in the project to confirm creation or inspect existing tasks.

In [30]:
for task in CurationTask.list(project_id=PROJECT_ENT.id, synapse_client=syn):
    print(f"  task_id={task.task_id}  data_type={task.data_type}  created_on={task.created_on}")


  task_id=2380  data_type=ClinicalFile_Staging_Curation  created_on=2026-03-02T23:35:09.000Z
  task_id=3142  data_type=ClincalFile_Staging_FileTask  created_on=2026-03-10T15:34:13.000Z
  task_id=3794  data_type=ClinicalFile_Staging_test_v1_FileTask  created_on=2026-04-08T22:13:02.000Z
  task_id=3795  data_type=ClinicalFile_Staging_test_v1_RecordTask  created_on=2026-04-08T22:15:40.000Z


### Work with a Grid Session Programmatically

Open a new Grid session on an existing RecordSet. After curators fill in the Grid via the Synapse UI, export it back to the RecordSet to trigger schema validation.

In [ ]:
record_set_obj = RecordSet(id=record_set.id).get(synapse_client=syn)
grid = run_grid_session(record_set_obj, syn)


In [ ]:
# After curators finish in the Synapse UI — export and close the session
grid.export_to_record_set(synapse_client=syn)
grid.delete(synapse_client=syn)
print("Grid exported and session closed.")


### Retrieve and Display Validation Results

After exporting the Grid, fetch detailed per-row validation results from the RecordSet.

In [29]:
record_set_obj = RecordSet(id=record_set.id).get(synapse_client=syn)
get_validation_report(record_set_obj)


[WARNING] 
!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
ifcollision=keep.both is being IGNORED because the download destination is synapse's cache. Instead, the behavior is "overwrite.local". 
!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!



[syn74385059:Job-7982620511909965807515593798.csv]: Found existing file at /home/ramayyala/.synapseCache/380/171112380/Job-7982620511909965807515593798.csv, skipping download.


AttributeError: 'RecordSet' object has no attribute 'get_schema_validation_statistics'